# Cost-Complexity Pruning in Decision Trees

---

## Why Pruning Is Needed

A decision tree can be grown until every leaf is pure or contains very few samples. Such a **fully grown tree**:

- Fits the training data extremely well (often zero training error).
- Captures not only the true signal but also the **noise** in the training set.
- Typically **overfits** and performs poorly on new, unseen data.

One option is to **stop early** while growing the tree, for example by:

- Limiting maximum depth,
- Requiring a minimum number of samples per leaf,
- Stopping when impurity reduction is small.

However, these early-stopping rules are **greedy** and local. They decide at the time of splitting, without seeing the long-term effect of later splits. This can cause the algorithm to stop **too early**, missing useful structure that only becomes clear deeper in the tree.

**Pruning** takes a different approach:

1. First, grow a **large tree** that fits the training data well.
2. Then, **simplify** it by removing subtrees that do not justify their complexity.

This leads to a **cost-complexity tradeoff**:

- Cost: how much training error (or impurity) the tree has.
- Complexity: how large and flexible the tree is (how many leaves).

We formalize this tradeoff with a cost-complexity criterion.

---

## Cost-Complexity Criterion

Let $T$ be a decision tree used for classification.

- Let $\mathcal{L}(T)$ be the set of **leaf nodes** of $T$.
- For each leaf $t \in \mathcal{L}(T)$, define the **misclassification cost** $R(t)$ as the number (or proportion) of misclassified samples in that leaf if it predicts the majority class of that leaf.
- Define the **total misclassification cost** of the tree:
  $$
  R(T) = \sum_{t \in \mathcal{L}(T)} R(t).
  $$

Let $|T|$ denote the **number of leaves** in tree $T$:
$$
|T| = |\mathcal{L}(T)|.
$$

We introduce a **complexity parameter** $\alpha \ge 0$ and define the **cost-complexity** of tree $T$ as

$$
\boxed{
R_\alpha(T) = R(T) + \alpha\,|T|.
}
$$

Interpretation:

- $R(T)$ measures how well the tree fits the training data (lower is better).
- $|T|$ measures how complex the tree is (larger means more flexibility).
- $\alpha$ is the **penalty per leaf**:
  - If $\alpha = 0$, there is no penalty for complexity, and we prefer the tree with **smallest training error** — usually the fully grown tree.
  - As $\alpha$ increases, extra leaves become more expensive, so **smaller trees** are preferred.
  - For very large $\alpha$, the optimal tree becomes extremely small (even a stump or single leaf).

So **$\alpha$ controls how aggressively we prune**:
- Small $\alpha$ → keep most of the tree.
- Large $\alpha$ → prune heavily.

---

## The Weakest Link and Effective Alpha

To prune in a principled way, we consider **collapsing** subtrees.

Take an **internal node** $t$ in tree $T$. Let:

- $T_t$ be the **subtree** rooted at node $t$ (all descendants of $t$).
- $|T_t|$ be the number of **leaves** in that subtree.
- $R(T_t)$ be the **total misclassification cost** of the subtree $T_t$:
  $$
  R(T_t) = \sum_{s \in \mathcal{L}(T_t)} R(s).
  $$
- $R(t)$ be the misclassification cost if we **collapse** $T_t$ into a single leaf at node $t$, predicting the majority class at $t$.

Collapsing $T_t$ into a leaf:

- Reduces the number of leaves by $(|T_t| - 1)$.
- **Increases** training misclassification cost from $R(T_t)$ to $R(t)$.

The **effective alpha** for node $t$ is defined as

$$
\boxed{
\alpha_{\text{eff}}(t)
=
\frac{R(t) - R(T_t)}{|T_t| - 1}.
}
$$

Interpretation:

- Numerator $R(t) - R(T_t)$:
  - How much the **training misclassification cost increases** when we replace the subtree $T_t$ with a single leaf at $t$.
- Denominator $|T_t| - 1$:
  - How many **leaves we remove** by collapsing $T_t$ into a single leaf.

So $\alpha_{\text{eff}}(t)$ is:

> The **increase in misclassification cost per leaf removed** when we collapse subtree $T_t$ into a leaf at $t$.

The **weakest link** in the tree is the internal node $t$ with the **smallest** effective alpha:

- It costs the **least** (per leaf) to remove that subtree, in terms of increased misclassification.
- That subtree is the “most expendable” part of the tree.

---

## The Weakest-Link Pruning Algorithm

The weakest-link pruning algorithm builds a sequence of nested trees by iteratively removing the weakest link.

**Step-by-step:**

1. **Start** with a fully grown tree $T_0$.
   - This tree is as large as allowed by your growing procedure (often very deep, near-zero training error).

2. **Compute effective alpha** for all internal nodes of $T_0$:
   $$
   \alpha_{\text{eff}}(t) = \frac{R(t) - R(T_t)}{|T_t| - 1}.
   $$

3. **Find the smallest** effective alpha:
   $$
   \alpha_1 = \min_{t} \alpha_{\text{eff}}(t).
   $$
   - Identify all internal nodes that attain this minimum.

4. **Collapse** all subtrees whose $\alpha_{\text{eff}}(t) = \alpha_1$:
   - Each of these subtrees $T_t$ is replaced by a single leaf at node $t$.
   - This gives a **smaller tree** $T_1$.

5. **Repeat**:
   - On $T_1$, recompute $\alpha_{\text{eff}}(t)$ for the remaining internal nodes.
   - Find the next smallest effective alpha $\alpha_2$.
   - Collapse all subtrees with $\alpha_{\text{eff}}(t) = \alpha_2$ to obtain $T_2$.
   - Continue until the tree has only the root node (one leaf). Call this $T_M$.

This process yields a **sequence** of nested trees:

$$
T_0 \supset T_1 \supset T_2 \supset \dots \supset T_M,
$$

and an associated sequence of increasing complexity parameters:

$$
0 = \alpha_0 < \alpha_1 < \alpha_2 < \dots < \alpha_M.
$$

For each interval $\alpha \in [\alpha_k, \alpha_{k+1})$, the tree $T_k$ is the tree that minimizes $R_\alpha(T) = R(T) + \alpha |T|$.

---

## Selecting Alpha via Cross-Validation

The weakest-link algorithm gives us a **pruning path** — a set of candidate trees:

$$
T_0, T_1, T_2, \dots, T_M.
$$

Each tree corresponds to some range of $\alpha$ values.

To choose the **best** tree (and thus the best $\alpha$), we usually use **$k$-fold cross-validation**:

1. Split the training data into $k$ folds.
2. For each fold:
   - Train a fully grown tree on the other $(k-1)$ folds.
   - Generate the pruning sequence for that tree: $T_0, T_1, \dots, T_M$.
   - Evaluate each pruned tree on the held-out fold.
3. Average the validation error for each level of pruning across folds.
4. Select the tree $T_k$ (or corresponding $\alpha_k$) that has the **lowest average validation error** (or use a 1-standard-error rule for more robustness).

**Why not pick the tree with lowest training error?**

- Training error always **decreases** or stays the same as the tree gets larger.
- The smallest training error is achieved by the **largest tree** (usually $T_0$), which is overfitted and has high variance.
- We care about **generalization** to unseen data, so we use validation error (approximation to test error) rather than training error.

Cross-validation balances the cost-complexity tradeoff in a data-driven way and picks the tree size that generalizes best.

---

## Numerical Example: Effective Alpha by Hand

Consider a very small tree:

- Root node splits into two internal nodes: $A$ and $B$.
- Each of $A$ and $B$ has two leaf children.

So the full tree $T_0$ has:

- 2 internal nodes ($A$ and $B$),
- 4 leaves: $A_1, A_2, B_1, B_2$.

Assume the per-leaf misclassification costs are:

- Leaves under $A$:
  - $R(A_1) = 3$
  - $R(A_2) = 1$
- Leaves under $B$:
  - $R(B_1) = 2$
  - $R(B_2) = 2$

So the total cost of the full tree is:

$$
R(T_0) = 3 + 1 + 2 + 2 = 8.
$$

Now suppose:

- At node $A$, if we collapse its subtree into a single leaf predicting the majority class at $A$, the misclassification cost is:
  $$
  R(A) = 5.
  $$
- At node $B$, if we collapse its subtree into a single leaf predicting the majority class at $B$, the misclassification cost is:
  $$
  R(B) = 7.
  $$

We also know:

- Subtree $T_A$ has 2 leaves: $|T_A| = 2$.
- Subtree $T_B$ has 2 leaves: $|T_B| = 2$.
- Their subtree costs are:
  $$
  R(T_A) = R(A_1) + R(A_2) = 3 + 1 = 4,
  $$
  $$
  R(T_B) = R(B_1) + R(B_2) = 2 + 2 = 4.
  $$

### Effective Alpha for Node A

Using
$$
\alpha_{\text{eff}}(t) = \frac{R(t) - R(T_t)}{|T_t| - 1},
$$
we have for $A$:

$$
\alpha_{\text{eff}}(A)
= \frac{R(A) - R(T_A)}{|T_A| - 1}
= \frac{5 - 4}{2 - 1}
= \frac{1}{1}
= 1.
$$

Interpretation:

- Collapsing subtree $T_A$ into a single leaf:
  - Increases misclassification cost by $5 - 4 = 1$.
  - Removes $2 - 1 = 1$ leaf.
- So the cost increase per leaf removed is $1$.

### Effective Alpha for Node B

Similarly for $B$:

$$
\alpha_{\text{eff}}(B)
= \frac{R(B) - R(T_B)}{|T_B| - 1}
= \frac{7 - 4}{2 - 1}
= \frac{3}{1}
= 3.
$$

Interpretation:

- Collapsing $T_B$:
  - Increases cost by $7 - 4 = 3$.
  - Removes $2 - 1 = 1$ leaf.
- Cost increase per leaf removed is $3$.

### Which Subtree Is Pruned First?

Compare the effective alphas:

- $\alpha_{\text{eff}}(A) = 1$
- $\alpha_{\text{eff}}(B) = 3$

The **weakest link** is node $A$ because it has the **smallest** effective alpha. Collapsing $A$:

- Has the smallest cost increase per leaf removed.
- Is the most efficient pruning step.

So:

1. For $\alpha < 1$, the full tree $T_0$ (with both $A$ and $B$ subtrees) is preferred.
2. At $\alpha = 1$, subtree $T_A$ becomes **too expensive** relative to its complexity, so it is collapsed to a single leaf, yielding tree $T_1$.
3. For $\alpha$ between $1$ and $3$, $T_1$ is optimal.
4. At $\alpha = 3$, subtree $T_B$ would be the next candidate for collapse, giving a smaller tree $T_2$.

This toy example illustrates exactly how **effective alpha** guides which parts of the tree are pruned first.

---m